3. Applied Data Science Level (Country Metadata Extraction)
Extract demographic/cultural metadata for European countries to enrich our core project pipeline:

Target Data Source:

Scrape tabular data from an open web source containing European nation details (e.g., Wikipedia/EU portal tables).

Extract at least 3 structural variables per country:

Country Name / ISO Code (e.g., ES, DE, FR).

Primary Official Language or Capital City.

Geographic / Cultural Region (e.g., Western Europe, Southern Europe, Nordic).

Data Cleaning & Export:

Strip reference markers (e.g., [1], [note A]) using regular expressions (re.sub(r'\[.*?\]', '', text)).

Align country names or codes to match the standard geo / CNTRY keys used in Eurostat and ESS microdata.

Export the cleaned DataFrame to data/processed/scraped_country_metadata.csv.

In [1]:


from pathlib import Path
import os
import pandas as pd
import random
#PATHING SCRIPT FOR EVERY EXCERSISE - PROJECT - WORKSPACE
# 1.Literal definition of route pathings(Every user of remote repository must config this pathing in order to find the local repository of his computer)
# My case:
ROOT = Path("/home/josu/Documentos/DataScienceCourse")

# We verify the existence before continue
if not ROOT.exists():
    raise FileNotFoundError(f"❌ The route {ROOT} doesn't exist. Check it.")

# 2. Fix the workspace
os.chdir(ROOT)
print(f"✅ Worskspace enabled!: {os.getcwd()}")

# 3. Define relative pathings to work properly
DATA_TABLES = ROOT / "data" / "raw" / "Tables"

# Let's verify our table folder:
if not DATA_TABLES.exists():
    # If it fails, monitorize the issue: 
    data_dir = ROOT / "data"
    if data_dir.exists():
        print(f"⚠️ the folder 'data' exists but it doesn't have any 'Tables'. 'data' content: {os.listdir(data_dir)}")
    else:
        print(f"⚠️ The folder 'data' doesn't exist on ROOT workspace: {os.listdir(ROOT)}")
    raise FileNotFoundError(f"❌ The tables route {DATA_TABLES} is missing.")

print(f"📂 Tables route detected: {DATA_TABLES}")


✅ Worskspace enabled!: /home/josu/Documentos/DataScienceCourse
📂 Tables route detected: /home/josu/Documentos/DataScienceCourse/data/raw/Tables


In [2]:
import pandas as pd
import duckdb

In [4]:
#Lets filter our main global csv data block to get solo the lines that pass our boolean mask , comparing it with
#An ISO-2 european country list:
#We will make an easy filter, instead to load 2 dataframes with this mask filter
#We force on csv reading, via duckdb the cols we want and lines with exact values we need.
#Duckdb is not only a DB manipulator, it gives us some tricks on csv or other tables reading.

'''
europe_iso2 = [
    'AL', 'AD', 'AT', 'BY', 'BE', 'BA', 'BG', 'HR', 'CY', 'CZ', 
    'DK', 'EE', 'FI', 'FR', 'DE', 'GR', 'HU', 'IS', 'IE', 'IT', 
    'XK', 'LV', 'LI', 'LT', 'LU', 'MT', 'MD', 'MC', 'ME', 'NL', 
    'MK', 'NO', 'PL', 'PT', 'RO', 'RU', 'SM', 'RS', 'SK', 'SI', 
    'ES', 'SE', 'CH', 'UA', 'GB', 'VA'
]
'''

dfEu = duckdb.sql("""
SELECT *
FROM read_csv('data/raw/Tables/GlobalCountriesLang.csv',
    header = true,
    delim = ','
)
WHERE Código IN (
    'AL','AD','AT','BY','BE','BA','BG','HR','CY','CZ',
    'DK','EE','FI','FR','DE','GR','HU','IS','IE','IT',
    'XK','LV','LI','LT','LU','MT','MD','MC','ME','NL',
    'MK','NO','PL','PT','RO','RU','SM','RS','SK','SI',
    'ES','SE','CH','UA','GB','VA'
)
""").df()
dfEu.head(5)

,column0,Código,Nombre del país,Countries,Official language(s),National language(s),Regional language(s),Minority language(s),Widely spoken
0,0,AD,Andorra,Andorra,Catalan[6],NaN,NaN,Spanish French Portuguese,NaN
1,4,AL,Albania,Albania,Albanian,NaN,NaN,Greek Macedonian Aromanian,Italian
2,8,AT,Austria,Austria,German,German (state language),Burgenland Croatian (parts of Burgenland) Hung...,Slovene Czech Hungarian Slovak Romani Serbian,English
3,11,BA,Bosnia y Herzegovina,Bosnia and Herzegovina,"None (Bosnian, Croatian and Serbian all have d...",NaN,NaN,NaN,NaN
4,14,BE,Bélgica,Belgium,Dutch French German,NaN,"Dutch (Flanders, Brussels)[16] French (Brussel...",NaN,English


In [ ]:
#As we can see, line 7 is incorrect(index positon = 8), 
#the text parser failed due to some unconventional html structure, 
#we have to delete that row but first adding the item value of line 7 to Description of line 6, 
#after that we will reset the index and we will devise a logical mask to mix the dataframes 
#and have valuable and compact information about European countries.
# Add row 8 item into row 7 description
dfRegions.loc[6, "Description"] = dfRegions.loc[7, "Item"]
# Delete row 8
dfRegions = dfRegions.drop(index=7)
# Reset index
dfRegions = dfRegions.reset_index(drop=True)
#Lets introduce manually the countries of mediterranean, macaronesia, and black sea region.
#The parser is not perfect.
macaronesia= 'Portugal and Spain'
black_sea= 'Bulgaria, Georgia, Romania, Russia, Turkey and Ukraine'
mediterranean = 'Albania, Bosnia and Herzegovina, Croatia, Cyprus, France, Greece, Italy, Malta, Monaco, Montenegro, Slovenia, Spain, Vatican City'
dfRegions.loc[18, "Description"] = macaronesia,
dfRegions.loc[19, "Description"] = mediterranean,
dfRegions.loc[20, "Description"] = black_sea,
dfRegions.head(10)

In [ ]:
#Lets create  a compressed dataframe for organize the geographical information in order to group item categoric
#Data in to Solo country name as index.
#The objective is to summ in one columm the peninsula and regional item information for every Europe Country

dfEuNodes = pd.Dataframe(['Countries','Iso2EU','GeoNodes'], axis=1)